In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import sys

In [3]:
sys.path.append('../src')

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

In [4]:
# Load processed data with risk scores (from Notebook 4)
df = pd.read_csv('../data/processed/complaints_with_risk_scores.csv')

print(f"Loaded: {len(df):,} complaints")
print(f"Date range: {df['Date received'].min()} to {df['Date received'].max()}")

# Summary of previous work
print("\nPrevious Analysis Summary:")
print(f"  Notebook 1: Data exploration ({len(df):,} complaints)")
print(f"  Notebook 2: Cleaned and preprocessed")
print(f"  Notebook 3: Extracted {len(df.columns)} features via TF-IDF")
print(f"  Notebook 4: Risk classification:")
print(f"    - High: {(df['risk_level']=='High').sum():,} ({(df['risk_level']=='High').mean()*100:.1f}%)")
print(f"    - Medium: {(df['risk_level']=='Medium').sum():,}")
print(f"    - Low: {(df['risk_level']=='Low').sum():,}")

Loaded: 1,049,446 complaints
Date range: 2025-01-20 to 2026-01-14

Previous Analysis Summary:
  Notebook 1: Data exploration (1,049,446 complaints)
  Notebook 2: Cleaned and preprocessed
  Notebook 3: Extracted 27 features via TF-IDF
  Notebook 4: Risk classification:
    - High: 8,690 (0.8%)
    - Medium: 357,265
    - Low: 683,491


In [6]:
# sentiment analysis
def analyze_sentiment(text):
    # textBlob
    if pd.isna(text) or text == '':
        return 0.0, 0.0
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

sentiments = df['complaint_clean'].apply(analyze_sentiment)
df['sentiment_polarity'] = sentiments.apply(lambda x: x[0])
df['sentiment_subjectivity'] = sentiments.apply(lambda x: x[1])

In [7]:
# Classify sentiment
df['sentiment_category'] = pd.cut(
    df['sentiment_polarity'],
    bins=[-1, -0.3, 0.3, 1],
    labels=['Negative', 'Neutral', 'Positive']
)

print("\nSentiment Distribution:")
print(df['sentiment_category'].value_counts())


Sentiment Distribution:
sentiment_category
Neutral     979594
Negative     35384
Positive     34411
Name: count, dtype: int64


In [8]:
# Sentiment-aware scoring
def calculate_sentiment_aware_score(row):
    base_score = row['emphasis_score']
    sentiment = row['sentiment_polarity']
    
    if sentiment > 0.3:
        multiplier = 0.3
    elif sentiment > 0:
        multiplier = 0.7
    else:
        multiplier = 1.0
    
    return base_score * multiplier

df['sentiment_aware_score'] = df.apply(calculate_sentiment_aware_score, axis=1)

In [9]:
# Re-classify
def classify_risk_sentiment_aware(score):
    if score >= 0.6:
        return 'High'
    elif score >= 0.3:
        return 'Medium'
    else:
        return 'Low'

df['risk_level_sentiment_aware'] = df['sentiment_aware_score'].apply(classify_risk_sentiment_aware)

# Compare
print("\nClassification Comparison:")
comparison = pd.crosstab(df['risk_level'], df['risk_level_sentiment_aware'], margins=True)
print(comparison)

changed = df[df['risk_level'] != df['risk_level_sentiment_aware']]
print(f"\nReclassified: {len(changed):,} complaints ({len(changed)/len(df)*100:.1f}%)")


Classification Comparison:
risk_level_sentiment_aware  High     Low  Medium      All
risk_level                                               
High                        4996     159    3535     8690
Low                            0  683491       0   683491
Medium                         0  131844  225421   357265
All                         4996  815494  228956  1049446

Reclassified: 135,538 complaints (12.9%)


In [11]:
# No response and untimely response analysis
# Create labels
def create_response_labels(df):
    no_response_values = ['In progress', 'No response', '', 'nan', None]
    df['no_response'] = df['Company response to consumer'].isin(no_response_values) | df['Company response to consumer'].isna()
    df['untimely_response'] = df['Timely response?'] == 'No'
    df['response_risk'] = (df['no_response'] | df['untimely_response']).astype(int)
    return df

df = create_response_labels(df)

print(f"No response: {df['no_response'].sum():,} ({df['no_response'].mean()*100:.2f}%)")
print(f"Response risk: {df['response_risk'].sum():,} ({df['response_risk'].mean()*100:.2f}%)")

No response: 395 (0.04%)
Response risk: 9,893 (0.94%)


In [12]:
# Build model
feature_cols = ['emphasis_score', 'sentiment_polarity', 'caps_words', 
                'exclamations', 'urgent_keywords', 'word_count']

X = df[feature_cols].fillna(0)
y = df['response_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("\nModel Performance:")
print(classification_report(y_test, y_pred))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)



Model Performance:
              precision    recall  f1-score   support

           0       1.00      0.66      0.80    207911
           1       0.02      0.71      0.04      1979

    accuracy                           0.66    209890
   macro avg       0.51      0.69      0.42    209890
weighted avg       0.99      0.66      0.79    209890


Feature Importance:
              feature  importance
5          word_count    0.337482
1  sentiment_polarity    0.314999
0      emphasis_score    0.124151
4     urgent_keywords    0.105303
2          caps_words    0.093319
3        exclamations    0.024746


In [14]:
df.head()

,Date received,year,month,month_name,day_of_week,Product,Sub-product,Issue,Sub-issue,Company,State,Company response to consumer,Timely response?,Consumer disputed?,complaint_clean,char_count,word_count,sentence_count,is_truncated,caps_words,exclamations,questions,repeated_punct,urgent_keywords,emphasis_score,risk_level,is_high_risk,sentiment_polarity,sentiment_subjectivity,sentiment_category,sentiment_aware_score,risk_level_sentiment_aware,no_response,untimely_response,response_risk
0,2025-12-28,2025,12,December,Sunday,Debt Collection,Telecommunications debt,Attempts to collect debt not owed,Debt was result of identity theft,"I.C. SYSTEM, INC.",IN,Closed with explanation,Yes,NaN,I do not recognize the aforementioned accounts...,826,129,6,False,1,0,0,False,4,0.325,Medium,False,-0.081378,0.277551,Neutral,0.3250,Medium,False,False,0
1,2025-09-26,2025,9,September,Friday,Credit Reporting Or Other Personal Consumer Re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"EQUIFAX, INC.",LA,Closed with non-monetary relief,Yes,NaN,I am writing pursuant to the Fair Credit Repor...,1054,180,10,False,4,0,0,False,3,0.400,Medium,False,-0.067361,0.486111,Neutral,0.4000,Medium,False,False,0
2,2025-03-15,2025,3,March,Saturday,Checking Or Savings Account,Checking account,Managing an account,Deposits and withdrawals,TD BANK US HOLDING COMPANY,MA,Closed with explanation,Yes,NaN,I am writing to formally dispute several unaut...,1124,172,9,False,1,0,0,False,3,0.325,Medium,False,0.080000,0.275000,Neutral,0.2275,Low,False,False,0
3,2025-05-03,2025,5,May,Saturday,Debt Collection,Other debt,Attempts to collect debt not owed,Debt was result of identity theft,"PORTFOLIO RECOVERY ASSOCIATES, LLC",TX,Closed with non-monetary relief,Yes,NaN,I am sending this asking for your help to file...,829,149,5,False,0,0,0,False,4,0.300,Medium,False,-0.022917,0.533333,Neutral,0.3000,Medium,False,False,0
4,2025-04-29,2025,4,April,Tuesday,Credit Reporting Or Other Personal Consumer Re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NJ,Closed with non-monetary relief,Yes,NaN,"Upon reviewing my consumer reports, l discover...",704,109,5,False,0,0,0,False,0,0.000,Low,False,-0.018750,0.510417,Neutral,0.0000,Low,False,False,0


Emphasis Score (Emphasis-Aware Score):
-  Measures HOW URGENTLY someone is writing
- Based on: CAPS, !!!, urgent words like "fraud", "lawyer"
- Range: 0 to 1
- Example: "FRAUD!!! STOLEN!!!" = High emphasis (0.85)

Sentiment Polarity:
- Measures EMOTIONAL TONE (positive/negative/neutral)
- Based on: Word meanings and context
- Range: -1 (very negative) to +1 (very positive)
- Example: "FRAUD!!! STOLEN!!!" = Very negative (-0.9)